# 04 — Network Analysis

Constructs and analyzes a multi-layer network of causal attributions (Layer 1) and emotions (Layer 2).

**Input:** `reddit_estrangement_pooled_emotions.csv`  
**Output:** Network metrics CSV, HTML visualizations, publication-quality figures

In [ ]:
!pip install networkx pyvis python-louvain --quiet

In [ ]:
# Setup
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import networkx as nx
from itertools import combinations
import community.community_louvain as community_louvain
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("/content/drive/MyDrive/Raw Data/reddit_estrangement_pooled_emotions.csv")
print(f"Loaded: {len(df)} users | {len(df.columns)} columns")

In [ ]:
# Define node lists
# Note: toxic_generic excluded (n=18, insufficient data)
CATEGORY_NODES = [
    'abuse', 'family_conflict', 'deceit', 'divorce_remarriage',
    'substance_use', 'ingratitude_entitlement', 'toxic_cruelty',
    'toxic_disrespect', 'toxic_anger', 'toxic_boundary_violations',
    'voluntary_distance', 'involuntary_distance',
    'parents_partner', 'childs_partner', 'mental_health',
    'selfishness', 'disapproval', 'exclusion', 'emotional_neglect',
    'material_deprivation', 'unknown_reason'
]

EMOTION_NODES = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval',
    'caring', 'confusion', 'curiosity', 'disappointment', 'disapproval',
    'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude',
    'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride',
    'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral'
]

THRESHOLD = 0.3

print(f"Category nodes: {len(CATEGORY_NODES)}")
print(f"Emotion nodes: {len(EMOTION_NODES)}")

In [ ]:
# Build Layer 1 — Causal Attribution Network
# Layer 1: Causal Attribution Network
G_categories = nx.Graph()

for cat in CATEGORY_NODES:
    freq = int(df[cat].sum())
    G_categories.add_node(cat, layer='category', frequency=freq)

print("Computing category co-occurrences...")
for cat1, cat2 in combinations(CATEGORY_NODES, 2):
    co_occur = int(((df[cat1] == 1) & (df[cat2] == 1)).sum())
    if co_occur > 0:
        union = int(((df[cat1] == 1) | (df[cat2] == 1)).sum())
        jaccard = co_occur / union if union > 0 else 0
        G_categories.add_edge(cat1, cat2, weight=co_occur, jaccard=round(jaccard, 4))

print(f"Category network: {G_categories.number_of_nodes()} nodes, {G_categories.number_of_edges()} edges")

In [ ]:
# Build Layer 2 — Emotion Network
# Layer 2: Emotion Network
G_emotions = nx.Graph()

for em in EMOTION_NODES:
    col = f"max_{em}"
    if col in df.columns:
        freq = int((df[col] >= THRESHOLD).sum())
        G_emotions.add_node(em, layer='emotion', frequency=freq)

print("Computing emotion co-occurrences...")
for em1, em2 in combinations(EMOTION_NODES, 2):
    col1, col2 = f"max_{em1}", f"max_{em2}"
    if col1 in df.columns and col2 in df.columns:
        co_occur = int(((df[col1] >= THRESHOLD) & (df[col2] >= THRESHOLD)).sum())
        if co_occur > 0:
            union = int(((df[col1] >= THRESHOLD) | (df[col2] >= THRESHOLD)).sum())
            jaccard = co_occur / union if union > 0 else 0
            G_emotions.add_edge(em1, em2, weight=co_occur, jaccard=round(jaccard, 4))

print(f"Emotion network: {G_emotions.number_of_nodes()} nodes, {G_emotions.number_of_edges()} edges")

In [ ]:
# Build Inter-layer — Bipartite Network
# Inter-layer: Bipartite Network
G_bipartite = nx.Graph()

for cat in CATEGORY_NODES:
    G_bipartite.add_node(cat, layer='category', frequency=int(df[cat].sum()))
for em in EMOTION_NODES:
    col = f"max_{em}"
    if col in df.columns:
        freq = int((df[col] >= THRESHOLD).sum())
        G_bipartite.add_node(em, layer='emotion', frequency=freq)

print("Computing category-emotion co-occurrences...")
for cat in CATEGORY_NODES:
    for em in EMOTION_NODES:
        col = f"max_{em}"
        if col in df.columns:
            co_occur = int(((df[cat] == 1) & (df[col] >= THRESHOLD)).sum())
            if co_occur > 0:
                cat_total = int(df[cat].sum())
                cond_prob = co_occur / cat_total if cat_total > 0 else 0
                G_bipartite.add_edge(cat, em, weight=co_occur, conditional_prob=round(cond_prob, 4))

print(f"Bipartite network: {G_bipartite.number_of_nodes()} nodes, {G_bipartite.number_of_edges()} edges")

In [ ]:
# Compute network metrics
def compute_metrics(G, name):
    degree_cent  = nx.degree_centrality(G)
    between_cent = nx.betweenness_centrality(G, weight='weight')
    weighted_deg = dict(G.degree(weight='weight'))

    freq_dict = {n: G.nodes[n].get('frequency', 0) for n in G.nodes()}

    df_metrics = pd.DataFrame({
        'node'             : list(degree_cent.keys()),
        'degree_centrality': list(degree_cent.values()),
        'betweenness'      : [between_cent[n] for n in degree_cent.keys()],
        'weighted_degree'  : [weighted_deg[n] for n in degree_cent.keys()],
        'frequency'        : [freq_dict[n] for n in degree_cent.keys()],
    }).sort_values('degree_centrality', ascending=False).reset_index(drop=True)

    print(f"\n{'='*55}")
    print(f"{name} METRICS")
    print(f"{'='*55}")
    print(df_metrics.head(10).to_string())
    return df_metrics

metrics_cat = compute_metrics(G_categories, "CATEGORY NETWORK")
metrics_em  = compute_metrics(G_emotions,   "EMOTION NETWORK")

In [ ]:
# Community detection — Louvain algorithm
print("Category network — Community Detection (Louvain):")
partition_cat = community_louvain.best_partition(G_categories, weight='weight')
communities_cat = {}
for node, comm_id in partition_cat.items():
    communities_cat.setdefault(comm_id, []).append(node)
for comm_id, nodes in sorted(communities_cat.items()):
    print(f"  Cluster {comm_id}: {nodes}")

print("\nEmotion network — Community Detection (Louvain):")
partition_em = community_louvain.best_partition(G_emotions, weight='weight')
communities_em = {}
for node, comm_id in partition_em.items():
    communities_em.setdefault(comm_id, []).append(node)
for comm_id, nodes in sorted(communities_em.items()):
    print(f"  Cluster {comm_id}: {nodes}")

In [ ]:
# Save metrics
metrics_cat['network'] = 'category'
metrics_em['network']  = 'emotion'
df_all_metrics = pd.concat([metrics_cat, metrics_em], ignore_index=True)

metrics_path = "/content/drive/MyDrive/Raw Data/network_metrics.csv"
df_all_metrics.to_csv(metrics_path, index=False, encoding="utf-8-sig")
print(f"✓ Metrics saved: {metrics_path}")

In [ ]:
# Interactive visualizations — PyVis
from pyvis.network import Network

COMMUNITY_COLORS = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A',
                    '#F4A261', '#A8DADC', '#6D6875', '#B5E48C']

def visualize_network(G, partition, title, filename):
    net = Network(height="700px", width="100%", bgcolor="#1a1a2e",
                  font_color="white", notebook=True)
    net.barnes_hut(gravity=-8000, central_gravity=0.3,
                   spring_length=150, spring_strength=0.05)

    max_freq   = max([G.nodes[n].get('frequency', 1) for n in G.nodes()])
    max_weight = max([d.get('weight', 1) for _, _, d in G.edges(data=True)] or [1])

    for node in G.nodes():
        freq  = G.nodes[node].get('frequency', 1)
        comm  = partition.get(node, 0)
        color = COMMUNITY_COLORS[comm % len(COMMUNITY_COLORS)]
        size  = 10 + (freq / max_freq) * 40
        net.add_node(node, label=node, color=color, size=size,
                     title=f"{node}\nFrequency: {freq}\nCluster: {comm}")

    for u, v, data in G.edges(data=True):
        w = data.get('weight', 1)
        net.add_edge(u, v, width=0.5 + (w / max_weight) * 3,
                     color="rgba(255,255,255,0.2)")

    net.set_options('''{"physics": {"stabilization": {"iterations": 200}}}''')
    net.show(filename)
    print(f"✓ Saved: {filename}")

visualize_network(G_categories, partition_cat, "Causal Attribution Network", "network_categories.html")
visualize_network(G_emotions,   partition_em,  "Emotion Network",            "network_emotions.html")

In [ ]:
# Publication figure — 2D multi-layer visualization (APA format)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import numpy as np

BG_COLOR     = '#FFFFFF'
CLUSTER0_CAT = '#2A9D8F'
CLUSTER1_CAT = '#1D3557'
INTER_EDGE   = '#E76F51'
CAT_EDGE     = '#2A9D8F'
EMO_EDGE     = '#E63946'
TEXT_COLOR   = '#111111'

EMO_CLUSTER_COLORS = {0: '#E9C46A', 1: '#E63946', 2: '#457B9D', 3: '#F4A261', 4: '#6D6875'}

def format_label(name):
    replacements = {
        'toxic_disrespect': 'Toxic Disrespect', 'toxic_cruelty': 'Toxic Cruelty',
        'toxic_anger': 'Toxic Anger', 'toxic_boundary_violations': 'Toxic Boundary Violations',
        'family_conflict': 'Family Conflict', 'emotional_neglect': 'Emotional Neglect',
        'material_deprivation': 'Material Deprivation', 'substance_use': 'Substance Use',
        'divorce_remarriage': 'Divorce/Remarriage', 'ingratitude_entitlement': 'Ingratitude/Entitlement',
        'voluntary_distance': 'Voluntary Distance', 'involuntary_distance': 'Involuntary Distance',
        'parents_partner': "Parent's Partner", 'childs_partner': "Child's Partner",
        'mental_health': 'Mental Health', 'unknown_reason': 'Unknown Reason',
        'disappointment': 'Disappointment', 'nervousness': 'Nervousness',
        'embarrassment': 'Embarrassment', 'realization': 'Realization',
        'admiration': 'Admiration', 'amusement': 'Amusement', 'annoyance': 'Annoyance',
        'approval': 'Approval', 'curiosity': 'Curiosity', 'confusion': 'Confusion',
        'gratitude': 'Gratitude', 'optimism': 'Optimism', 'surprise': 'Surprise',
        'remorse': 'Remorse', 'sadness': 'Sadness', 'neutral': 'Neutral',
        'disgust': 'Disgust', 'caring': 'Caring', 'relief': 'Relief',
        'anger': 'Anger', 'fear': 'Fear', 'grief': 'Grief', 'pride': 'Pride',
        'abuse': 'Abuse', 'deceit': 'Deceit', 'exclusion': 'Exclusion',
        'selfishness': 'Selfishness', 'disapproval': 'Disapproval',
        'excitement': 'Excitement', 'love': 'Love', 'joy': 'Joy',
    }
    return replacements.get(name, name.replace('_', ' ').title())

np.random.seed(42)

CATEGORY_NODES_SORTED = sorted(
    CATEGORY_NODES,
    key=lambda n: (partition_cat.get(n, 0), -G_categories.nodes[n]['frequency'])
)
emo_nodes_present = [e for e in EMOTION_NODES if f"max_{e}" in df.columns]
EMO_NODES_SORTED = sorted(
    emo_nodes_present,
    key=lambda n: (partition_em.get(n, 0), -int((df[f"max_{n}"] >= THRESHOLD).sum()))
)

n_cat = len(CATEGORY_NODES_SORTED)
pos_cat = {node: (0.0, i / (n_cat - 1)) for i, node in enumerate(CATEGORY_NODES_SORTED)}
n_emo = len(EMO_NODES_SORTED)
pos_emo = {node: (1.0, i / (n_emo - 1)) for i, node in enumerate(EMO_NODES_SORTED)}

fig, ax = plt.subplots(figsize=(24, 18), facecolor=BG_COLOR)
ax.set_facecolor(BG_COLOR)
ax.set_xlim(-0.35, 1.35)
ax.set_ylim(-0.05, 1.08)
ax.axis('off')

max_cat_w = max([d['weight'] for _, _, d in G_categories.edges(data=True)])
for u, v, data in G_categories.edges(data=True):
    if u not in pos_cat or v not in pos_cat:
        continue
    xu, yu = pos_cat[u]; xv, yv = pos_cat[v]
    lw = 0.2 + (data['weight'] / max_cat_w) * 1.2
    alpha = 0.08 + (data['weight'] / max_cat_w) * 0.25
    ax.annotate("", xy=(xv, yv), xytext=(xu, yu),
                arrowprops=dict(arrowstyle="-", color=CAT_EDGE, alpha=alpha,
                                lw=lw, connectionstyle="arc3,rad=-0.3"))

max_em_w = max([d['weight'] for _, _, d in G_emotions.edges(data=True)])
for u, v, data in G_emotions.edges(data=True):
    if u in pos_emo and v in pos_emo:
        xu, yu = pos_emo[u]; xv, yv = pos_emo[v]
        lw = 0.2 + (data['weight'] / max_em_w) * 1.2
        alpha = 0.08 + (data['weight'] / max_em_w) * 0.25
        ax.annotate("", xy=(xv, yv), xytext=(xu, yu),
                    arrowprops=dict(arrowstyle="-", color=EMO_EDGE, alpha=alpha,
                                    lw=lw, connectionstyle="arc3,rad=0.3"))

all_inter = [(u, v, d) for u, v, d in G_bipartite.edges(data=True)
             if (u in pos_cat or v in pos_cat)]
max_inter_w = max([d['weight'] for _, _, d in all_inter]) if all_inter else 1
for u, v, data in all_inter:
    if u in pos_cat and v in pos_emo:
        xu, yu = pos_cat[u]; xv, yv = pos_emo[v]
    elif v in pos_cat and u in pos_emo:
        xu, yu = pos_cat[v]; xv, yv = pos_emo[u]
    else:
        continue
    lw = 0.15 + (data['weight'] / max_inter_w) * 1.5
    alpha = 0.08 + (data['weight'] / max_inter_w) * 0.5
    ax.plot([xu, xv], [yu, yv], color=INTER_EDGE, linewidth=lw, alpha=alpha, zorder=2)

max_cat_freq = max([G_categories.nodes[n]['frequency'] for n in CATEGORY_NODES])
for node, (x, y) in pos_cat.items():
    freq = G_categories.nodes[node]['frequency']
    size = 60 + (freq / max_cat_freq) * 380
    comm = partition_cat.get(node, 0)
    color = CLUSTER0_CAT if comm == 0 else CLUSTER1_CAT
    ax.scatter(x, y, s=size, c=color, edgecolors='white', linewidths=0.8, zorder=5, marker='o')
    ax.text(x - 0.04, y, format_label(node), fontsize=7.5, color=TEXT_COLOR,
            ha='right', va='center', fontweight='bold', zorder=6)

max_em_freq = max([(df[f"max_{n}"] >= THRESHOLD).sum() for n in emo_nodes_present])
for node, (x, y) in pos_emo.items():
    freq = int((df[f"max_{node}"] >= THRESHOLD).sum())
    size = 60 + (freq / max_em_freq) * 380
    comm = partition_em.get(node, 0)
    color = EMO_CLUSTER_COLORS.get(comm, '#888888')
    ax.scatter(x, y, s=size, c=color, edgecolors='white', linewidths=0.8, zorder=5, marker='D')
    ax.text(x + 0.04, y, format_label(node), fontsize=7.5, color=TEXT_COLOR,
            ha='left', va='center', fontweight='bold', zorder=6)

ax.text(0.0, 1.06, "Layer 1: Estrangement Categories", fontsize=11, color=TEXT_COLOR, fontweight='bold', ha='center')
ax.text(1.0, 1.06, "Layer 2: Emotions (ATME)",         fontsize=11, color=TEXT_COLOR, fontweight='bold', ha='center')

legend_elements = [
    mpatches.Patch(color=CLUSTER0_CAT, label='Category Cluster 0 (Overt Harm)'),
    mpatches.Patch(color=CLUSTER1_CAT, label='Category Cluster 1 (Relational Dynamics)'),
    mpatches.Patch(color='#E63946',    label='Emotion Cluster 1 (Negative Appraisal)'),
    mpatches.Patch(color='#457B9D',    label='Emotion Cluster 2 (Positive/Social)'),
    mpatches.Patch(color='#F4A261',    label='Emotion Cluster 3 (Threat-Related)'),
    mlines.Line2D([0],[0], color=CAT_EDGE,   lw=2, alpha=0.5, label='Category Co-occurrence'),
    mlines.Line2D([0],[0], color=EMO_EDGE,   lw=2, alpha=0.5, label='Emotion Co-occurrence'),
    mlines.Line2D([0],[0], color=INTER_EDGE, lw=2, alpha=0.8, label='Inter-layer Connection'),
]
ax.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, -0.04), ncol=4,
          facecolor='white', edgecolor='#CCCCCC', labelcolor='black', fontsize=8, framealpha=1)

plt.tight_layout()
plt.savefig("Figure1_multilayer_network.png", dpi=300, bbox_inches='tight', facecolor=BG_COLOR)
plt.show()
print("✓ Saved: Figure1_multilayer_network.png")
print()
print("Figure 1")
print("Multi-Layer Network of Estrangement Causal Attributions and Emotions")
print("Note. Layer 1 (left) = causal attribution network (n=21, 210 edges);")
print("Layer 2 (right) = emotion network (n=27, 276 edges). Node size = frequency.")
print("Node color = community. Orange lines = inter-layer connections (n=47 nodes, 507 edges).")
print("toxic_generic excluded (n=18).")